# VIDSGEN Phase 5A: Kaggle GPU preflight (read-only)

This notebook **only inspects** the Kaggle environment so we can plan Wan 2.2 TI2V-5B inference safely.

**It does NOT:** download model weights · install packages · start ComfyUI · run inference · open tunnels · print secrets.

**Before running:**
1. *Settings → Accelerator:* **GPU T4 x2** (the plan uses **one** GPU; T4 VRAM is not pooled).
2. *Settings → Internet:* **On** (phone-verified Kaggle account required).
3. *Optional:* *Add-ons → Secrets* → `HF_TOKEN` (read-only token). Not needed for these public repos; it is never printed.
4. **Run All.** It takes about 1–2 minutes.

**When finished:** download `/kaggle/working/vidsgen_preflight_report.json` (right sidebar → Output) or copy the
summary printed by the last cell, and send it back. It contains no tokens.

In [ ]:
# --- 1. Settings and helpers -------------------------------------------------------------
import json, os, platform, shutil, socket, subprocess, sys, time
from pathlib import Path

VIDSGEN_REPO   = "https://github.com/Prosperotechit/VIDSGEN.git"
VIDSGEN_COMMIT = "4b6fc67243b53cb2dcfb7d2b114dca2517380176"   # Phase 4 (validated workflows + downloader)
COMFYUI_TAG    = "v0.46.0"                                     # proposed pin (Phase 5A report, section E)
COMFYUI_TAG_COMMIT = "a277b5bf213e8a44fa34817b7770b54f86ef4cea"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

# Storage budget (bytes). Model sizes come from the repo manifest (cell 2); the rest are planning allowances.
GB = 1024**3
BUDGET = {
    "comfyui_code":           0.2 * GB,   # ComfyUI checkout at a pinned tag (~80 MB) + headroom
    "python_packages":        3.0 * GB,   # ComfyUI requirements not preinstalled on Kaggle (installed with --no-cache-dir)
    "cloudflared":            0.1 * GB,
    "outputs_and_uploads":    1.0 * GB,   # generated MP4s + I2V start images during testing
    "comfyui_temp_and_cache": 1.0 * GB,
}
SAFETY_FRACTION = 0.10                    # +10 % of everything, minimum 2 GB

REPORT = {"generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "checks": [], "facts": {}}

def check(name, status, detail):
    """status: PASS | WARN | FAIL | INFO"""
    REPORT["checks"].append({"name": name, "status": status, "detail": detail})
    print(f"[{status:^4}] {name}: {detail}")

def gb(n):
    return f"{n / GB:.1f} GiB" if n is not None else "n/a"

def run(cmd, timeout=60):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout or "") + (p.stderr or "")
    except Exception as e:
        return -1, f"{type(e).__name__}: {e}"

print("Working directory:", WORK)

In [ ]:
# --- 2. Reuse the VIDSGEN repo (pinned Phase 4 commit): model manifest + downloader --------
REPO_DIR = WORK / "vidsgen-src"
if not (REPO_DIR / ".git").exists():
    code, out = run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", VIDSGEN_REPO, str(REPO_DIR)], timeout=180)
    if code != 0:
        check("vidsgen_repo", "FAIL", "could not clone the VIDSGEN repo: " + out.strip()[-200:])
code, out = run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", VIDSGEN_COMMIT])
head = run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"])[1].strip()
if head == VIDSGEN_COMMIT:
    check("vidsgen_repo", "PASS", f"checked out pinned commit {head[:12]}")
else:
    check("vidsgen_repo", "FAIL", f"expected {VIDSGEN_COMMIT[:12]}, got {head[:12] or 'nothing'}")

MANIFEST = None
try:
    import yaml
    MANIFEST = yaml.safe_load((REPO_DIR / "workflows/comfyui/models.yaml").read_text())
    MODEL_BYTES = sum(f["size_bytes"] for f in MANIFEST["files"])
    check("model_manifest", "PASS", f"{len(MANIFEST['files'])} files, {MODEL_BYTES:,} bytes ({gb(MODEL_BYTES)}) from models.yaml")
except Exception as e:
    MODEL_BYTES = 18_144_966_705   # Phase 4 manifest total, used only if the repo could not be read
    check("model_manifest", "WARN", f"could not read models.yaml ({type(e).__name__}); using Phase 4 total {gb(MODEL_BYTES)}")
REPORT["facts"]["model_bytes"] = MODEL_BYTES

In [ ]:
# --- 3. Environment and package versions -----------------------------------------------
from importlib import metadata

# Only these variables are shown. Kaggle also sets secret-bearing variables
# (e.g. KAGGLE_USER_SECRETS_TOKEN), which are never printed or saved.
SAFE_ENV = ["KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_DOCKER_IMAGE", "KAGGLE_CONTAINER_NAME", "KAGGLE_DATA_PROXY_PROJECT",
            "CUDA_VERSION", "NVIDIA_DRIVER_CAPABILITIES", "CUDA_VISIBLE_DEVICES", "PYTHONPATH"]
env = {k: os.environ.get(k) for k in SAFE_ENV if os.environ.get(k)}
facts = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "is_kaggle": Path("/kaggle").exists(), "env": env,
}
pkgs = ["torch", "torchvision", "torchaudio", "xformers", "numpy", "safetensors", "transformers", "tokenizers",
        "sentencepiece", "einops", "scipy", "pillow", "av", "aiohttp", "yarl", "requests", "pyyaml", "psutil",
        "huggingface_hub", "SQLAlchemy", "alembic", "kornia", "spandrel", "pydantic", "torchsde", "blake3"]
versions = {}
for p in pkgs:
    try:
        versions[p] = metadata.version(p)
    except metadata.PackageNotFoundError:
        versions[p] = None
facts["packages"] = versions
REPORT["facts"]["environment"] = facts
print(json.dumps({"python": facts["python"], "platform": facts["platform"], "env": env}, indent=1))
print("\nPackages (None = not preinstalled; ComfyUI would need to install it):")
for p, v in versions.items():
    print(f"  {p:<16} {v}")
py = tuple(int(x) for x in facts["python"].split(".")[:2])
check("python_version", "PASS" if py >= (3, 10) else "FAIL", f"Python {facts['python']} (ComfyUI v0.46.0 needs >= 3.10)")
check("kaggle_environment", "PASS" if facts["is_kaggle"] else "WARN",
      "running on Kaggle" if facts["is_kaggle"] else "not a Kaggle environment (/kaggle missing)")

In [ ]:
# --- 4. GPU, VRAM, CUDA and PyTorch compatibility --------------------------------------
gpu = {"nvidia_smi": None, "devices": []}
code, out = run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,driver_version,compute_cap,pci.bus_id",
                 "--format=csv,noheader"])
gpu["nvidia_smi"] = out.strip() if code == 0 else None
code2, banner = run(["nvidia-smi"])
driver_cuda = None
if code2 == 0:
    for line in banner.splitlines():
        if "CUDA Version" in line:
            driver_cuda = line.split("CUDA Version:")[1].split()[0].strip(" |")
print(out if code == 0 else "nvidia-smi not available: " + out.strip()[:200])

try:
    import torch
    gpu["torch"] = torch.__version__
    gpu["torch_cuda_build"] = torch.version.cuda
    gpu["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
    gpu["driver_cuda"] = driver_cuda
    gpu["cuda_available"] = torch.cuda.is_available()
    gpu["arch_list"] = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            gpu["devices"].append({"index": i, "name": p.name, "capability": f"{p.major}.{p.minor}",
                                   "total_bytes": total, "free_bytes": free, "multiprocessors": p.multi_processor_count})
        try:
            gpu["bf16_native"] = torch.cuda.is_bf16_supported(including_emulation=False)
        except TypeError:  # older PyTorch without the argument
            gpu["bf16_native"] = torch.cuda.is_bf16_supported()
        # Tiny smoke test on GPU 0 only (a 256x256 fp16 matmul; not inference).
        try:
            a = torch.randn(256, 256, device="cuda:0", dtype=torch.float16)
            ok = bool(torch.isfinite(a @ a).all().item())
            gpu["fp16_kernel_ok"] = ok
            del a
            torch.cuda.empty_cache()
        except Exception as e:
            gpu["fp16_kernel_ok"] = False
            gpu["fp16_kernel_error"] = f"{type(e).__name__}: {str(e)[:200]}"
except Exception as e:
    gpu["torch_error"] = f"{type(e).__name__}: {e}"
REPORT["facts"]["gpu"] = gpu

devs = gpu["devices"]
if not devs:
    check("gpu_detected", "FAIL", "no CUDA GPU visible to PyTorch (set Accelerator: GPU T4 x2)")
else:
    names = ", ".join(f"GPU{d['index']}: {d['name']} (sm_{d['capability'].replace('.', '')}, {gb(d['total_bytes'])})" for d in devs)
    check("gpu_detected", "PASS", names)
    d0 = devs[0]
    check("single_gpu_plan", "INFO",
          f"{len(devs)} GPU(s) visible. VRAM is NOT pooled: ComfyUI will be pinned to GPU0 with "
          f"{gb(d0['total_bytes'])} total / {gb(d0['free_bytes'])} free")
    vram = d0["total_bytes"]
    check("gpu0_vram", "PASS" if vram >= 15 * GB else "WARN" if vram >= 12 * GB else "FAIL",
          f"{gb(vram)} on GPU0 (fp16 diffusion model alone is ~9.3 GiB; ComfyUI offloads the text encoder)")
    cap = d0["capability"].replace(".", "")
    in_arch = any(a.endswith("_" + cap) for a in gpu.get("arch_list", []))
    check("torch_supports_gpu_arch", "PASS" if in_arch else "FAIL",
          f"sm_{cap} {'is' if in_arch else 'is NOT'} in this PyTorch build's arch list {gpu.get('arch_list')}")
    check("fp16_kernel", "PASS" if gpu.get("fp16_kernel_ok") else "FAIL",
          "small fp16 matmul ran on GPU0" if gpu.get("fp16_kernel_ok") else gpu.get("fp16_kernel_error", "failed"))
    check("bf16_support", "INFO", f"native bf16: {gpu.get('bf16_native')} (T4/Turing has none: fp16 will be used)")
tc, dc = gpu.get("torch_cuda_build"), gpu.get("driver_cuda")
if tc and dc:
    t, d = tuple(map(int, tc.split(".")[:2])), tuple(map(int, dc.split(".")[:2]))
    check("cuda_driver_compat", "PASS" if t <= d else "WARN",
          f"PyTorch built for CUDA {tc}; driver supports up to CUDA {dc}"
          + ("" if t <= d else " (driver older than build: relies on CUDA minor-version compatibility)"))
else:
    check("cuda_driver_compat", "FAIL" if devs else "INFO", f"PyTorch CUDA build {tc}, driver CUDA {dc}")
check("pytorch", "INFO", f"torch {gpu.get('torch')}, CUDA build {tc}, cuDNN {gpu.get('cudnn')}")

In [ ]:
# --- 5. System RAM ------------------------------------------------------------------------
mem = {}
with open("/proc/meminfo") as f:
    for line in f:
        k, v = line.split(":", 1)
        mem[k] = int(v.split()[0]) * 1024
ram_total, ram_avail = mem.get("MemTotal", 0), mem.get("MemAvailable", 0)
REPORT["facts"]["ram"] = {"total_bytes": ram_total, "available_bytes": ram_avail, "swap_bytes": mem.get("SwapTotal", 0)}
check("system_ram", "PASS" if ram_total >= 28 * GB else "WARN" if ram_total >= 20 * GB else "FAIL",
      f"{gb(ram_total)} total, {gb(ram_avail)} available, swap {gb(mem.get('SwapTotal', 0))} "
      "(ComfyUI keeps offloaded weights in RAM: ~9.3 GiB model + ~6.3 GiB text encoder)")

In [ ]:
# --- 6. Disk space and storage budget ----------------------------------------------------
candidates = ["/kaggle/working", "/kaggle/tmp", "/tmp", "/root", "/kaggle/input"]
disks, seen = [], {}
for path in candidates:
    p = Path(path)
    if not p.exists():
        continue
    st = os.statvfs(path)
    total, free = st.f_blocks * st.f_frsize, st.f_bavail * st.f_frsize
    dev = os.stat(path).st_dev
    entry = {"path": path, "free_bytes": free, "total_bytes": total, "device": dev,
             "same_filesystem_as": seen.get(dev), "writable": os.access(path, os.W_OK)}
    seen.setdefault(dev, path)
    disks.append(entry)
code, df = run(["df", "-h", *[d["path"] for d in disks]])
print(df)

need_extra = sum(BUDGET.values())
subtotal = MODEL_BYTES + need_extra
margin = max(SAFETY_FRACTION * subtotal, 2 * GB)
required = subtotal + margin
REPORT["facts"]["storage"] = {"disks": disks, "budget_bytes": BUDGET, "model_bytes": MODEL_BYTES,
                              "required_bytes": required, "safety_margin_bytes": margin}
print(f"Required for models + ComfyUI + packages + outputs + {SAFETY_FRACTION:.0%} margin: {gb(required)} "
      f"(models {gb(MODEL_BYTES)}, other {gb(need_extra)}, margin {gb(margin)})")

best = None
for d in disks:
    share = f" (same filesystem as {d['same_filesystem_as']})" if d["same_filesystem_as"] else ""
    if not d["writable"]:
        check(f"disk {d['path']}", "INFO", f"read-only{share}")
        continue
    ok = d["free_bytes"] >= required
    if d["path"] == "/kaggle/working":
        check("disk /kaggle/working", "PASS" if ok else "WARN",
              f"{gb(d['free_bytes'])} free of {gb(d['total_bytes'])}{share}. It is persisted as notebook output and "
              f"size-capped; do not put model weights here" + ("" if ok else f" (needs {gb(required)})"))
        continue
    check(f"disk {d['path']}", "PASS" if ok else "FAIL",
          f"{gb(d['free_bytes'])} free of {gb(d['total_bytes'])}{share}; needs {gb(required)}")
    if ok and (best is None or d["free_bytes"] > best["free_bytes"]):
        best = d
REPORT["facts"]["recommended_install_dir"] = best["path"] + "/vidsgen-gpu" if best else None
check("storage_plan", "PASS" if best else "FAIL",
      f"install ComfyUI + models under {best['path']}/vidsgen-gpu (non-persistent, enough space)" if best
      else "no writable non-persistent location has enough free space for the 18.1 GB of models plus overhead")

In [ ]:
# --- 7. Internet connectivity (no downloads) -----------------------------------------------
import requests
UA = {"User-Agent": "vidsgen-preflight"}
targets = {
    "huggingface.co":   "https://huggingface.co/api/models/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/revision/main",
    "github.com":       "https://github.com/comfyanonymous/ComfyUI/releases/tag/" + COMFYUI_TAG,
    "pypi.org":         "https://pypi.org/simple/comfyui-frontend-package/",
    "cloudflared release": "https://github.com/cloudflare/cloudflared/releases/latest",
    "trycloudflare.com": "https://api.trycloudflare.com/",
}
net = {}
for name, url in targets.items():
    t0 = time.time()
    try:
        r = requests.get(url, headers=UA, timeout=15, allow_redirects=False, stream=True)
        r.close()                      # headers only: the body is never read
        net[name] = {"status": r.status_code, "ms": int((time.time() - t0) * 1000)}
        code = r.status_code
        # 2xx/3xx, or 404/405 (the server answered) = reachable; 401/403/429 = reachable but refused.
        status = "PASS" if code < 400 or code in (404, 405) else "WARN" if code in (401, 403, 429) else "FAIL"
    except Exception as e:
        net[name] = {"error": type(e).__name__}
        status = "FAIL"
    if name == "trycloudflare.com" and status == "FAIL":
        status = "WARN"   # only needed in Phase 5B; reported, not blocking
    check(f"internet {name}", status, json.dumps(net[name]))
REPORT["facts"]["network"] = net

In [ ]:
# --- 8. Hugging Face metadata: canonical repo, pinned commit, sizes, SHA-256 (no downloads) --
# The SHA-256 of a Git-LFS file on Hugging Face is its LFS object id. It is read here from two
# independent metadata sources and cross-checked: (a) the paths-info API and (b) the X-Linked-Etag
# header of a HEAD request on the resolve URL with redirects disabled (no file content is transferred).
hf_headers = dict(UA)
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    if _tok:
        hf_headers["Authorization"] = f"Bearer {_tok}"
    del _tok
    print("HF_TOKEN secret: attached (value not shown)")
except Exception:
    print("HF_TOKEN secret: not attached (fine for these public repositories)")

hf = {"repos": {}, "files": []}
files = (MANIFEST or {}).get("files", [])
if not files:
    check("hf_metadata", "FAIL", "no manifest files to check (see cell 2)")
for f in files:
    repo, path, expected = f["repo"], f["path"], f["size_bytes"]
    item = {"role": f["role"], "manifest_repo": repo, "path": path, "manifest_size": expected}
    try:
        if repo not in hf["repos"]:
            r = requests.get(f"https://huggingface.co/api/models/{repo}/revision/main", headers=hf_headers, timeout=30)
            r.raise_for_status()
            j = r.json()
            hf["repos"][repo] = {"canonical_id": j.get("id"), "main_commit": j.get("sha")}
        rev = hf["repos"][repo]["main_commit"]
        item.update(hf["repos"][repo])
        r = requests.post(f"https://huggingface.co/api/models/{repo}/paths-info/{rev}",
                          data={"paths": path, "expand": "true"}, headers=hf_headers, timeout=30)
        r.raise_for_status()
        info = (r.json() or [{}])[0]
        lfs = info.get("lfs") or {}
        item["api_size"], item["api_sha256"] = lfs.get("size") or info.get("size"), lfs.get("oid")
        h = requests.head(f"https://huggingface.co/{repo}/resolve/{rev}/{path}", headers=hf_headers,
                          timeout=30, allow_redirects=False)
        item["head_size"] = int(h.headers["x-linked-size"]) if h.headers.get("x-linked-size") else None
        item["head_sha256"] = (h.headers.get("x-linked-etag") or "").strip('"') or None
        sizes_ok = item["api_size"] == expected == item["head_size"]
        sha_ok = bool(item["api_sha256"]) and item["api_sha256"] == item["head_sha256"] and len(item["api_sha256"]) == 64
        item["sizes_match_manifest"], item["sha256_cross_checked"] = sizes_ok, sha_ok
        renamed = item["canonical_id"] and item["canonical_id"] != repo
        check(f"hf {path.split('/')[-1]}", "PASS" if sizes_ok and sha_ok else "FAIL",
              f"size {item['api_size']:,} ({'matches' if sizes_ok else 'DIFFERS FROM'} manifest), "
              f"sha256 {item['api_sha256']} ({'confirmed by 2 sources' if sha_ok else 'NOT cross-checked'}), "
              f"commit {str(rev)[:12]}" + (f", repo now named {item['canonical_id']}" if renamed else ""))
    except Exception as e:
        item["error"] = f"{type(e).__name__}: {str(e)[:200]}"
        check(f"hf {path.split('/')[-1]}", "FAIL", item["error"])
    hf["files"].append(item)
hf_headers.pop("Authorization", None)
REPORT["facts"]["huggingface"] = hf

In [ ]:
# --- 9. Existing downloader: DRY RUN only (never --execute) ----------------------------------
install_dir = REPORT["facts"].get("recommended_install_dir") or "/tmp/vidsgen-gpu"
args = [sys.executable, str(REPO_DIR / "scripts/download_models.py"), "--comfyui-dir", install_dir + "/ComfyUI"]
assert "--execute" not in args, "the preflight must never download weights"
code, out = run(args, timeout=60)
print(out)
REPORT["facts"]["downloader_dry_run"] = out[-3000:]
check("downloader_dry_run", "PASS" if code == 0 and "DRY RUN: nothing was downloaded" in out else "FAIL",
      "existing scripts/download_models.py ran in dry-run mode; nothing was downloaded" if code == 0
      else "dry run failed: " + out.strip()[-200:])
weights_present = list(Path(install_dir).rglob("*.safetensors")) if Path(install_dir).exists() else []
check("no_weights_downloaded", "PASS" if not weights_present else "FAIL",
      "no .safetensors files exist under " + install_dir if not weights_present else f"unexpected files: {weights_present[:3]}")

In [ ]:
# --- 10. Summary and report ---------------------------------------------------------------------
counts = {s: sum(c["status"] == s for c in REPORT["checks"]) for s in ("PASS", "WARN", "FAIL", "INFO")}
REPORT["summary"] = counts
out_path = WORK / "vidsgen_preflight_report.json"
import re
text = json.dumps(REPORT, indent=1, default=str)
# Defence in depth: the report must never contain a token.
assert "Bearer " not in text and not re.search(r"hf_[A-Za-z0-9]{30,}", text), "token-like text found; report not written"
out_path.write_text(text)
print("=" * 78)
print(f"VIDSGEN Phase 5A preflight: {counts['PASS']} PASS, {counts['WARN']} WARN, {counts['FAIL']} FAIL, {counts['INFO']} INFO")
print("=" * 78)
for c in REPORT["checks"]:
    if c["status"] != "PASS":
        print(f"[{c['status']}] {c['name']}: {c['detail']}")
print(f"\nFull report: {out_path}  (no secrets). Send this file, or this output, back for review.")

## What happens next (not in this notebook)
Phase 5B (only after review): install ComfyUI at the pinned tag, download the three model files with SHA-256
verification, start ComfyUI on `127.0.0.1` only, and add the token-protected proxy + Cloudflare quick tunnel.